# Amazon Bedrock curso - Interface Gráfica com Streamlit

Anotações do curso de Amazon Bedrock sobre como transformar o RAG em uma **aplicação web de chat**: o usuário digita a pergunta em uma interface gráfica e recebe a resposta gerada a partir da base de conhecimento, com as fontes usadas.

A lógica de RAG vem do notebook **Amazon Bedrock curso - RAG** (seção 6); a forma de conectar à AWS vem dos notebooks anteriores.

## Sumário

1. [Configuração do ambiente](#1.-Configuração-do-ambiente)
2. [Como o Streamlit funciona](#2.-Como-o-Streamlit-funciona)
3. [Arquitetura da aplicação](#3.-Arquitetura-da-aplicação)
4. [Criando o `app.py`](#4.-Criando-o-app.py)
5. [Executando a interface](#5.-Executando-a-interface)
6. [Versão original do curso (referência)](#6.-Versão-original-do-curso-(referência))
7. [Resumo dos aprendizados](#7.-Resumo-dos-aprendizados)

---
## 1. Configuração do ambiente

O **Streamlit** é uma biblioteca que transforma um script Python em uma aplicação web, sem precisar escrever HTML, CSS ou JavaScript. É ideal para criar protótipos de chatbots e dashboards.

Vamos instalar o Streamlit e o Boto3 (que já usamos para conectar à AWS).

> 💡 O `%pip` instala no mesmo ambiente do kernel do notebook (mais seguro que `!pip`).

In [1]:
# Instalando o Streamlit (interface web) e o Boto3 (conexão com a AWS)
%pip install streamlit boto3

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.4 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.4 MB 1.5 MB/s eta 0:00:07
   ---- ----------------------------------- 1.0/10.4 MB 2.1 MB/s eta 0:00:05
   -------- ------------------------------- 2.1/10.4 MB 2.9 MB/s eta 0:00:03
   ------------ --------------------------- 3.1/10.4 MB 3.5 MB/s eta 0:00:03
   ---------------- ----------------------- 4.2/10.4 MB 3.7 MB/s eta 0:00:02
   -------------------- ------------------- 5.2/10.4 MB 3.9 MB/s eta 0:00:02
   ------------------------ --------------- 6.3/10.4 MB 4.0 MB/s eta 0:00:02
   ----------------------------- ---------- 7.6/10.4 MB 4.2 MB/s eta 0:00:01
   -------------------------------- ------- 8.4/10.4 MB 4.2 MB/s eta 0:00:01
   ------------------------------------ --- 9.4/10.4 MB 4.3 MB/s eta 0:00:01
   ---------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## 2. Como o Streamlit funciona

O ponto mais importante para entender o código: **a cada interação do usuário** (enviar uma pergunta, mexer em um slider, clicar em um botão), **o Streamlit executa o script inteiro de novo**, do início ao fim.

Isso traz duas consequências:

- **Variáveis comuns são perdidas** a cada reexecução. Para guardar o histórico da conversa usamos o **`st.session_state`**, um dicionário que sobrevive entre as execuções.
- **Recursos caros seriam recriados** toda vez (como os clientes da AWS). O decorador **`@st.cache_resource`** faz a função rodar só na primeira vez e reaproveitar o resultado.

### 2.1 Componentes usados

| Componente | Para que serve |
| --- | --- |
| `st.set_page_config` | Título da aba, ícone e layout da página. Deve ser o primeiro comando do Streamlit. |
| `st.title` / `st.caption` | Título e subtítulo da página. |
| `st.session_state` | Guarda dados entre as reexecuções (aqui, o histórico de mensagens). |
| `st.cache_resource` | Cria um recurso uma única vez e reaproveita (aqui, os clientes da AWS). |
| `st.sidebar` | Barra lateral com as configurações. |
| `st.slider` / `st.button` | Controles para ajustar parâmetros e limpar a conversa. |
| `st.chat_message` | Balão de mensagem do chat (`"user"` ou `"assistant"`). |
| `st.chat_input` | Campo de digitação fixo no rodapé da página. |
| `st.spinner` | Indicador de carregamento enquanto a resposta é gerada. |
| `st.expander` | Bloco que abre e fecha; usado para mostrar as fontes. |
| `st.error` | Caixa vermelha com mensagem de erro. |

### 2.2 O operador `:=` (walrus)

O código usa `if prompt := st.chat_input(...)`. O `:=` atribui e testa ao mesmo tempo: guarda o texto digitado em `prompt` e só entra no `if` se o usuário tiver enviado alguma coisa. É equivalente a:

```python
prompt = st.chat_input("Digite sua pergunta")
if prompt:
    ...
```

---
## 3. Arquitetura da aplicação

### 3.1 Por que refatorar

Na aula, a interface chama uma **API do curso** (API Gateway → Lambda → Agente do Bedrock), criada na conta e na região do instrutor (`us-east-1`). Essa API não faz parte da nossa conta, então não temos controle sobre ela.

Por isso a interface foi refatorada para usar **os recursos que criamos e testamos** nos notebooks anteriores:

| Item | Origem |
| --- | --- |
| Credenciais lidas do CSV + `boto3.Session` | Notebook **Modelos** (seção 1.1) |
| Knowledge Base `4SCLDZ9ZTH` (Bedrock Managed, `us-east-1`) | Notebook **RAG** (seções 4 e 5) |
| Função `ask`: `retrieve` + `converse` com Nova Micro | Notebook **RAG** (seção 6) |
| Fontes (citações) com arquivo, página e score | Notebook **RAG** (seções 7 e 9) |

### 3.2 Fluxo de uma pergunta

```
 Usuário            Streamlit (app.py)                     Amazon Bedrock
+--------+   (1)   +---------------------+   (2) retrieve   +----------------------+
|Pergunta| ------> |   ask(pergunta)     | ---------------> | Knowledge Base (PDFs)|
+--------+         |                     | <--- trechos --- +----------------------+
                   |  monta o prompt     |   (3) converse   +----------------------+
+--------+   (5)   |  com o contexto     | ---------------> | Nova Micro (LLM)     |
|Resposta| <------ |                     | <-- resposta --- +----------------------+
|+ fontes|         +---------------------+
+--------+               (4) mostra resposta + fontes e salva no histórico
```

1. O usuário digita a pergunta no `st.chat_input`.
2. **Retrieval:** `retrieve` busca os trechos mais relevantes na Knowledge Base.
3. **Generation:** os trechos viram o contexto do prompt, e o `converse` gera a resposta.
4. A interface mostra a resposta e as fontes, e salva tudo no `st.session_state`.
5. Na próxima interação, o histórico é reexibido a partir do `session_state`.

---
## 4. Criando o `app.py`

O Streamlit roda a partir de um **arquivo `.py`**, e não de um notebook. O comando mágico **`%%writefile app.py`** grava o conteúdo da célula em um arquivo, na mesma pasta do notebook (se o arquivo já existir, ele é sobrescrito).

O código está dividido em 5 blocos:

| Bloco | O que faz |
| --- | --- |
| **1. Configurações** | IDs da base e do modelo, região e caminho do CSV de credenciais. |
| **2. Conexão com a AWS** | `criar_clientes()` lê o CSV, cria a sessão e os dois clientes. Com `@st.cache_resource`, isso acontece uma única vez. |
| **3. RAG** | `ask()` faz a busca (`retrieve`) e a geração (`converse`). Se a busca não trouxer trechos, responde sem chamar o modelo. |
| **4. Fontes** | `mostrar_fontes()` exibe, em um `st.expander`, o arquivo, a página, o score e o texto de cada trecho. |
| **5. Interface** | Título, barra lateral (sliders e botão de limpar), histórico e campo de pergunta. |

> ⚠️ O caminho do CSV é montado com `Path(__file__).parent`, ou seja, **a pasta onde está o `app.py`**. Assim o app encontra as credenciais independentemente da pasta de onde o comando `streamlit run` for executado.

**Tratamento de erros:** a chamada à AWS fica dentro de um `try`. Se o CSV não existir (`FileNotFoundError`) ou a AWS recusar a chamada (`ClientError`, `BotoCoreError`), o usuário vê uma mensagem com `st.error`, em vez de a aplicação quebrar.

In [2]:
%%writefile app.py

import csv
from pathlib import Path

import boto3
import streamlit as st
from botocore.exceptions import BotoCoreError, ClientError

# ------------------------------------------------------------------
# 1. Configurações (as mesmas do notebook de RAG)
# ------------------------------------------------------------------
KNOWLEDGE_BASE_ID = "4SCLDZ9ZTH"
MODEL_ID = "us.amazon.nova-micro-v1:0"
REGION = "us-east-1"
# O CSV de credenciais fica na mesma pasta do app.py (fora do código)
ARQUIVO_CREDENCIAIS = Path(__file__).parent / "curso_bedrock_accessKeys.csv"

st.set_page_config(
    page_title="Assistente IA Expert Academy",
    page_icon="🤖",
    layout="wide"
)


# ------------------------------------------------------------------
# 2. Conexão com a AWS
# ------------------------------------------------------------------
# O Streamlit reexecuta o script a cada interação. O cache_resource
# faz os clientes serem criados uma única vez e reaproveitados.
@st.cache_resource
def criar_clientes():
    with open(ARQUIVO_CREDENCIAIS, encoding="utf-8-sig") as f:
        credenciais = next(csv.DictReader(f))

    session = boto3.Session(
        aws_access_key_id=credenciais["Access key ID"],
        aws_secret_access_key=credenciais["Secret access key"],
        region_name=REGION,
    )
    # bedrock-agent-runtime: busca na Knowledge Base | bedrock-runtime: geração da resposta
    return session.client("bedrock-agent-runtime"), session.client("bedrock-runtime")


# ------------------------------------------------------------------
# 3. RAG em duas etapas: busca + geração
# ------------------------------------------------------------------
def ask(question, n_resultados=5, temperature=0.2):
    bedrock_agent_runtime, bedrock_runtime = criar_clientes()

    # Retrieval: a base é "Bedrock Managed", por isso usa managedSearchConfiguration
    retrieval = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=KNOWLEDGE_BASE_ID,
        retrievalQuery={"text": question},
        retrievalConfiguration={
            "managedSearchConfiguration": {"numberOfResults": n_resultados}
        }
    )
    resultados = retrieval["retrievalResults"]

    # Sem trechos encontrados não há contexto, então nem chamamos o modelo
    if not resultados:
        return {
            "resposta": "Não encontrei informações sobre isso na base de conhecimento.",
            "fontes": []
        }

    contexto = "\n\n".join(r["content"]["text"] for r in resultados)

    # Generation: o modelo responde usando apenas o contexto recuperado
    prompt = (
        "Responda à pergunta usando somente as informações do contexto abaixo. "
        "Se a resposta não estiver no contexto, diga que não sabe.\n\n"
        f"Contexto:\n{contexto}\n\nPergunta: {question}"
    )
    response = bedrock_runtime.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 512, "temperature": temperature}
    )

    return {
        "resposta": response["output"]["message"]["content"][0]["text"],
        "fontes": resultados
    }


# ------------------------------------------------------------------
# 4. Exibição das fontes (citações)
# ------------------------------------------------------------------
def mostrar_fontes(fontes):
    if not fontes:
        return

    with st.expander(f"📄 Fontes ({len(fontes)} trechos)"):
        for i, fonte in enumerate(fontes, start=1):
            uri = (fonte.get("location") or {}).get("s3Location", {}).get("uri", "")
            arquivo = uri.rsplit("/", 1)[-1] if uri else "documento"
            pagina = (fonte.get("metadata") or {}).get("x-amz-bedrock-kb-document-page-number")
            score = fonte.get("score")

            titulo = f"**Trecho {i}** · {arquivo}"
            if pagina is not None:
                titulo += f" · página {int(float(pagina))}"
            if score is not None:
                titulo += f" · score {score:.3f}"

            st.markdown(titulo)
            st.caption(fonte["content"]["text"])


# ------------------------------------------------------------------
# 5. Interface
# ------------------------------------------------------------------
st.title("🤖 Assistente IA Expert Academy")
st.caption("Pergunte sobre cursos, trilhas, planos e termos de uso da IA Expert Academy.")

# O histórico fica no session_state para sobreviver às reexecuções do script
if "messages" not in st.session_state:
    st.session_state.messages = []

with st.sidebar:
    st.header("⚙️ Configurações")
    n_resultados = st.slider("Trechos buscados na base", min_value=1, max_value=10, value=5)
    temperature = st.slider("Temperature", min_value=0.0, max_value=1.0, value=0.2, step=0.1)

    if st.button("🗑️ Limpar conversa"):
        st.session_state.messages = []
        st.rerun()

    st.divider()
    st.caption(f"Base de conhecimento: `{KNOWLEDGE_BASE_ID}`  \nModelo: `{MODEL_ID}`")

# Reexibe o histórico da conversa
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])
        mostrar_fontes(message.get("fontes"))

if prompt := st.chat_input("Digite sua pergunta"):
    # Usuário
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # Assistente
    with st.chat_message("assistant"):
        with st.spinner("Consultando a base de conhecimento..."):
            try:
                resultado = ask(prompt, n_resultados=n_resultados, temperature=temperature)
            except FileNotFoundError:
                st.error(f"Arquivo de credenciais não encontrado: `{ARQUIVO_CREDENCIAIS.name}`")
            except (ClientError, BotoCoreError) as erro:
                st.error(f"Erro ao consultar a AWS: {erro}")
            else:
                st.markdown(resultado["resposta"])
                mostrar_fontes(resultado["fontes"])
                st.session_state.messages.append({
                    "role": "assistant",
                    "content": resultado["resposta"],
                    "fontes": resultado["fontes"]
                })

Writing app.py


### 4.1 Conferindo o arquivo

Antes de abrir a interface, verificamos se o `app.py` não tem erro de sintaxe. O `py_compile` só **compila** o arquivo, sem executá-lo, então não chama a AWS.

In [3]:
# Verificando se o app.py foi gravado sem erros de sintaxe (não executa o app)
import py_compile

py_compile.compile("app.py", doraise=True)
print("app.py sem erros de sintaxe")

app.py sem erros de sintaxe


---
## 5. Executando a interface

O Streamlit é um servidor web: ele fica rodando até ser interrompido. Por isso **não executamos pelo notebook** (a célula ficaria travada) e sim pelo **terminal**.

1. Abra um terminal no VS Code (`Ctrl + '`), na pasta do projeto.
2. Execute:

```bash
python -m streamlit run app.py
```

> ⚠️ Use `python -m streamlit` e não só `streamlit`. Quando o pacote é instalado no diretório do usuário (`AppData\Roaming\Python\...\Scripts`), essa pasta não fica no PATH do Windows e o terminal responde *"'streamlit' não é reconhecido como um comando interno"*. Chamando pelo `python -m`, o Python encontra o módulo direto.

3. O navegador abre em **http://localhost:8501**. Se não abrir, copie o endereço que aparece no terminal.
4. Para encerrar, volte ao terminal e pressione **`Ctrl + C`**.

> 💡 Ao salvar alterações no `app.py` com o servidor rodando, o Streamlit mostra o botão **Rerun** no canto superior direito da página para recarregar a aplicação.

### 5.1 Roteiro de teste

| Teste | O que observar |
| --- | --- |
| Perguntar *"Quais são as trilhas?"* | A resposta lista as trilhas, e o expander **Fontes** mostra os trechos do PDF. |
| Perguntar *"Qual o plano mais barato?"* | A resposta traz o Plano Mensal (R$ 24,90). |
| Perguntar algo fora do PDF (ex.: *"Qual a capital da França?"*) | O modelo deve dizer que não sabe, pois só pode usar o contexto. |
| Mudar o slider **Trechos buscados** | Mais trechos = mais contexto (e mais tokens de entrada). |
| Clicar em **Limpar conversa** | O histórico é apagado. |

### 5.2 Limitações

- **Sem memória entre perguntas:** o histórico aparece na tela, mas não é enviado ao modelo. Cada pergunta é respondida de forma independente, então perguntas como *"e a segunda?"* não funcionam.
- **Credenciais locais:** o app lê o CSV da pasta do projeto, o que serve para rodar no próprio computador. Para publicar a aplicação (ex.: Streamlit Community Cloud), as chaves devem ir para o `st.secrets`, e **o CSV nunca deve ser enviado para o GitHub**.
- **Custo:** cada pergunta faz uma busca na Knowledge Base e uma inferência no modelo.

---
## 6. Versão original do curso (referência)

Na aula, a interface envia a pergunta por HTTP para uma API (API Gateway + Lambda), que por sua vez chama um **Agente do Bedrock**. O código abaixo fica só como referência e **não é executado** neste notebook, porque a API pertence à conta do curso.

```python
import streamlit as st
import requests

API_URL = "https://7grol1mvv3.execute-api.us-east-2.amazonaws.com/agente_producao_ia/chamada_agente_ia"

# ... mesma estrutura de chat (session_state, chat_message, chat_input) ...

response = requests.post(API_URL, json={"pergunta": prompt})
if response.status_code == 200:
    resposta = response.json()["resposta"]
else:
    resposta = f"Erro {response.status_code}"
```

| | Original (curso) | Refatorada (este notebook) |
| --- | --- | --- |
| Como chega ao Bedrock | `requests.post` → API Gateway → Lambda → Agente | `boto3` direto: `retrieve` + `converse` |
| Recursos usados | Conta do curso, `us-east-2` | Nossa conta, `us-east-1` |
| Credenciais no app | Nenhuma (ficam na Lambda) | CSV local |
| Fontes na tela | Não | Sim (arquivo, página, score e texto) |
| Configurações na tela | Não | Trechos buscados, temperature, limpar conversa |

> 💡 A arquitetura com API tem uma vantagem para produção: as credenciais da AWS ficam no servidor, e o front-end não precisa conhecê-las. Para estudo e uso local, chamar o Bedrock direto é mais simples.

---
## 7. Resumo dos aprendizados

- O **Streamlit** transforma um script Python em aplicação web. Ele roda a partir de um arquivo `.py`, com `python -m streamlit run app.py` no terminal.
- O script é **reexecutado a cada interação**: use `st.session_state` para guardar o histórico e `@st.cache_resource` para criar os clientes da AWS uma única vez.
- Os componentes de chat são `st.chat_message` (balões) e `st.chat_input` (campo de pergunta); `st.spinner`, `st.expander` e `st.sidebar` completam a experiência.
- A interface reaproveita o RAG do notebook anterior: `retrieve` com `managedSearchConfiguration` + `converse` com o Nova Micro.
- Mostrar as **fontes** (arquivo, página, score) deixa a resposta verificável pelo usuário.
- Erros da AWS devem ser tratados com `try/except` e exibidos com `st.error`, para a aplicação não quebrar.
- Credenciais nunca vão para o código nem para o GitHub. Para publicar, use `st.secrets` ou uma API intermediária, como na versão original do curso.